# 실습 3 · 프로토타입 분류 — 9개 산업

장표 21(nearest-centroid). **클래스마다 예시 몇 개의 평균 = 프로토타입**, 새 글은 가장 가까운 프로토타입의 클래스로. 모델을 더 학습하지 않는다.

| ds | 산업 | 과제 | 데이터셋 |
|---|---|---|---|
| massive | 고객센터 | 음성비서 명령 → 의도 60종 (한국어) | mteb/amazon_massive_intent · ko |
| ynat | 언론 | 뉴스 제목 → 주제 7종 (한국어) | klue/klue · ynat |
| lbox | 법무 | 판례 사실관계 → 죄명 12종 (한국어) | lbox/lbox_open |
| osha | 제조 · 건설 안전 | 중대재해 경위 → 사고 유형 7종 | masonmarker/osha-severe-injury-reports-enriched |
| nhtsa | 제조 · 품질 | 차량 결함 신고 → 결함 부품 5종 | vic35get/nhtsa_complaints_dataset |
| product | 제조 · 유통 | 상품 설명 → 카테고리 33종 | Lezh1n/ecommerce-product-classification-by-categories |
| maib | 해운 · 물류 | 해양사고 보고 → 사고 유형 | baker-street/maib-incident-reports-5K |
| aviation | 항공 | 사고 원인 요약 → 원인 분류 8종 | himaxym/aviation-safety-occurrences |
| patent | 연구 · 특허 | 특허 초록 → 기술 부문 8종 | NortheasternUniversity/big_patent |

- 이 노트북만으로 끝난다 : 데이터셋마다 클래스당 예시 후보 `POOL` 개 + 시험 글 `TEST` 개만 임베딩(기본 4,317개 · 강사 PC 단독 약 20분)

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 준비 — DB · 임베딩 서버 · 도구 (01~04 공통) |
| 3 | 테이블 만들기 — `lab3.cls_items` · 뷰 `lab3.cls_proto`(프로토타입) |
| 4 | 받기 · 표 — 9개 데이터셋 → 클래스당 예시 후보 · 시험 글 |
| 5 | 적재 |
| 6 | 임베딩 |
| 7 | 불러오기 · 채점 도구 |
| 8 | 한 예시 — 고객 문의(한국어) |
| 9 | 분류 API — 새 문장을 프로토타입 뷰로 (SQL 한 문장) |
| 10 | 예시 수(1 · 3 · 5 · 10)별 정확도 — 9개 산업 |
| 11 | 코사인 vs 내적 — 클래스마다 응집도(‖M‖)가 다를 때 |
| 12 | "기타" 거르기 — 처음 보는 클래스 걸러내기 기준값 |
| 13 | 정리 |

### 셀 1 · 설정
- `DATASETS` : 쓸 데이터셋(빼면 그만큼 빨라진다) · `POOL` : 클래스당 예시 후보 수(예시 k 개를 여기서 뽑는다) · `TEST` : 클래스당 시험 글 수
- `POOL + TEST` 개가 안 되는 작은 클래스는 뺀다(해양사고의 드문 유형 등). massive 는 원래의 train 에서 예시 후보, test 에서 시험 글

In [ ]:
# 셀 1 · 설정
DB_URL       = "postgresql://lab:lab@db:5432/lab"       # 4-1 도커의 DB
LMSTUDIO_URL = "http://host.docker.internal:1234/v1"    # LM Studio 서버 주소 (4-1 실습1 셀 1 과 같게)
EMBED_MODEL  = "text-embedding-qwen3-embedding-8b"      # LM Studio 에 로드된 임베딩 모델 이름
CACHE        = "/cache/hf/lab3"                         # 받은 원본 저장 위치 = 도커 볼륨 hfcache
BATCH_SIZE   = 16                                       # 한 번에 LM Studio 로 보내는 글 수
MAX_CHARS    = 1500                                     # 이보다 긴 글은 잘라서 넣는다
SEED         = 42
RESET        = False    # True : 이 실습의 테이블을 지우고 다시 만든다(표본 크기를 바꿨을 때). 이미 계산한 벡터는 캐시에서 돌아온다

DATASETS = ["massive","ynat","lbox","osha","nhtsa","product","maib","aviation","patent"]
POOL   = 10              # 클래스당 예시 후보 (SHOTS 의 최댓값 이상)
TEST   = 20              # 클래스당 시험 글
SHOTS  = [1, 3, 5, 10]   # 클래스당 예시 수
REPEAT = 5               # 예시를 다시 뽑아 평균낼 횟수
INDUSTRY = {"massive": "고객센터", "ynat": "언론", "lbox": "법무", "osha": "제조 · 건설 안전", "nhtsa": "제조 · 품질",
            "product": "제조 · 유통", "maib": "해운 · 물류", "aviation": "항공", "patent": "연구 · 특허"}

### 셀 2 · 준비 — DB · 임베딩 서버 · 도구
01~04 실습의 셀 2 는 모두 같다(각 실습을 따로 돌릴 수 있게 노트북마다 들어 있다).
- DB 연결(4-1 과 같은 pgvector DB, 스키마 `lab3`) · LM Studio 확인
- **임베딩 캐시** `lab3.emb_cache` : 키 = (모델 이름, 글의 md5) → 벡터. 같은 글은 한 번만 계산한다(예 : 실습 3 에서 계산한 판례는 실습 4 에서 다시 계산하지 않음). 실습 테이블을 지워도 남고, 지우려면 `TRUNCATE lab3.emb_cache`
- `embed_table(테이블)` : 4-1 실습1 셀 7 과 같은 배치 임베딩. ① 캐시에서 먼저 채우고 ② 남은 행을 `BATCH_SIZE` 개씩 LM Studio 로 → `UPDATE` + 캐시 저장. 멈췄다 다시 실행하면 빈 행부터
- `load(SQL)` : 마지막 열이 벡터인 SELECT → (표, 벡터 행렬 · 길이 1)
- `ci(값들)` : 단위별 차이의 평균과 부트스트랩 95% 신뢰구간 — 표본이 작아 차이가 우연인지 함께 본다(구간이 0 을 포함하면 "차이가 있다고 말할 수 없음")

In [ ]:
# 셀 2 · 준비
import json, time, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
import requests
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)
C = Path(CACHE); (C / "raw").mkdir(parents=True, exist_ok=True)
HF = "https://huggingface.co/datasets"
rng = np.random.default_rng(SEED)

conn = psycopg.connect(DB_URL, autocommit=True)          # 문장마다 바로 반영 (여러 문장을 묶을 때만 conn.transaction())
register_vector(conn)
conn.execute("CREATE SCHEMA IF NOT EXISTS lab3")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.emb_cache (          -- 임베딩 캐시 (01~04 공용)
        model     text NOT NULL,                         -- 임베딩 모델 이름 (모델이 다르면 벡터도 다르다)
        text_md5  text NOT NULL,                         -- md5(글)
        embedding vector(4096) NOT NULL,
        PRIMARY KEY (model, text_md5))""")


def one(q, params=None):
    """SQL 결과의 첫 행 첫 값"""
    row = conn.execute(q, params).fetchone()
    return row[0] if row else None


def sql(q, params=None):
    """SQL 결과 → 표"""
    with conn.cursor() as cur:
        cur.execute(q, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def ci(x, n=1000):
    """단위(사용자 · 질의 · 글 …)별 값 x 의 평균과 부트스트랩 95% 신뢰구간. 구간이 0 을 포함하지 않으면 ✔ (차이가 우연이라고 보기 어려움)"""
    x = np.asarray(x, float); r = np.random.default_rng(SEED)
    b = [x[r.integers(0, len(x), len(x))].mean() for _ in range(n)]
    lo, hi = np.quantile(b, [.025, .975])
    return f"{x.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]" + (" ✔" if lo > 0 or hi < 0 else "")


def load(q, params=None):
    """마지막 열이 벡터인 SELECT → (표, 벡터 행렬 float32 · 길이 1). 이진 전송이라 벡터가 바로 numpy 로 온다"""
    with conn.cursor(binary=True) as cur:
        cur.execute(q, params)
        rows = cur.fetchall()
        names = [c.name for c in cur.description]
    assert rows and all(r[-1] is not None for r in rows), "임베딩이 빈 행이 있다 → 셀 6(임베딩)을 끝까지"
    return pd.DataFrame([r[:-1] for r in rows], columns=names[:-1]), unit(np.stack([r[-1] for r in rows]).astype(np.float32))


def download(url, name):
    """볼륨에 없을 때만 받는다(.part 로 받고 끝나면 이름 변경)"""
    p = C / "raw" / name
    if not p.exists():
        t = time.time(); tmp = p.with_suffix(p.suffix + ".part")
        urllib.request.urlretrieve(url, tmp); tmp.rename(p)
        print(f"  받음 {name} {p.stat().st_size / 1e6:.1f}MB · {time.time() - t:.0f}s")
    return p


def parquet_urls(ds, config, split):
    r = requests.get(f"https://huggingface.co/api/datasets/{ds}/parquet/{config}/{split}", timeout=60)
    r.raise_for_status(); return r.json()


def all_parquet_urls(ds, config="default"):
    """그 구성의 모든 분할(train · test …) parquet 주소"""
    r = requests.get(f"https://huggingface.co/api/datasets/{ds}/parquet", timeout=60); r.raise_for_status()
    return [u for us in r.json()[config].values() for u in us]


def clean(d):
    """글 다듬기 : NUL 제거 · 앞뒤 공백 · MAX_CHARS 에서 자르기"""
    d = d.reset_index(drop=True)
    d["text"] = d["text"].astype(str).str.replace("\x00", "", regex=False).str.strip().str.slice(0, MAX_CHARS)
    d["label"] = d["label"].astype(str)
    return d


def per_label(d, k):
    """라벨마다 k 개씩 무작위. 같은 SEED 면 작은 k 의 표본은 큰 k 표본의 앞부분이다(→ 캐시가 그대로 맞는다)"""
    return pd.concat([g.sample(min(len(g), k), random_state=SEED) for _, g in d.groupby("label")]).reset_index(drop=True)


def embed(texts):
    """LM Studio /v1/embeddings. 실패하면 5 · 10 · 20초 뒤 다시"""
    for wait in [5, 10, 20, None]:
        try:
            r = requests.post(f"{LMSTUDIO_URL}/embeddings", json={"model": EMBED_MODEL, "input": texts}, timeout=600)
            r.raise_for_status()
            return [np.asarray(d["embedding"], dtype=np.float32) for d in sorted(r.json()["data"], key=lambda d: d["index"])]
        except Exception as e:
            if wait is None: raise
            print(f"  ! 임베딩 실패({type(e).__name__}) → {wait}s 뒤 다시"); time.sleep(wait)


def embed_table(table):
    """빈 embedding 채우기 : ① 캐시(같은 모델 · 같은 글) → ② 남은 행을 BATCH_SIZE 개씩 LM Studio 로 · UPDATE + 캐시 저장"""
    t0 = time.time()
    hit = conn.execute(f"""UPDATE {table} t SET embedding = c.embedding FROM lab3.emb_cache c
                           WHERE t.embedding IS NULL AND c.model = %s AND c.text_md5 = md5(t.text)""", (EMBED_MODEL,)).rowcount
    todo = one(f"SELECT count(*) FROM {table} WHERE embedding IS NULL")
    print(f"[{table}] 캐시에서 {hit:,}건 · 새로 임베딩 {todo:,}건", flush=True)
    done, nb, t1 = 0, 0, time.time()
    while True:
        rows = conn.execute(f"SELECT id, text FROM {table} WHERE embedding IS NULL ORDER BY id LIMIT %s", (BATCH_SIZE,)).fetchall()
        if not rows:
            break
        vecs = embed([t for _, t in rows])
        with conn.transaction(), conn.cursor() as cur:
            cur.executemany(f"UPDATE {table} SET embedding = %s WHERE id = %s", [(v, i) for (i, _), v in zip(rows, vecs)])
            cur.executemany("INSERT INTO lab3.emb_cache VALUES (%s, md5(%s), %s) ON CONFLICT DO NOTHING",
                            [(EMBED_MODEL, t, v) for (_, t), v in zip(rows, vecs)])
        done += len(rows); nb += 1
        if nb % 25 == 0 or done >= todo:
            el = time.time() - t1; rate = done / el
            print(f"  {done:6,}/{todo:,}  {rate:5.1f}건/s  경과 {el / 60:5.1f}분  남은 시간 약 {max(todo - done, 0) / rate / 60:5.1f}분", flush=True)
    print(f"[{table}] 끝 · 캐시 {hit:,}건 + 새로 {done:,}건 · {time.time() - t0:.0f}s")


try:
    models = [m["id"] for m in requests.get(f"{LMSTUDIO_URL}/models", timeout=10).json()["data"]]
except requests.exceptions.RequestException:
    raise ConnectionError(f"LM Studio 에 연결할 수 없음 : {LMSTUDIO_URL}\n"
                          "  ① LM Studio 의 Developer 탭에서 서버가 켜져 있는지 확인\n"
                          "  ② 포트가 다르면 셀 1 의 LMSTUDIO_URL 을 고치고 셀 1 부터 다시") from None
assert EMBED_MODEL in models, f"셀 1 의 EMBED_MODEL 을 이 목록의 이름으로 바꾼다 : {models}"
print("pgvector", one("SELECT extversion FROM pg_extension WHERE extname = 'vector'"), "· 임베딩", embed(["확인"])[0].size, "차원",
      f"· 캐시 {one('SELECT count(*) FROM lab3.emb_cache WHERE model = %s', (EMBED_MODEL,)):,}건")

### 셀 3 · 테이블 만들기
- `lab3.cls_items` : 데이터셋(`ds`) · 라벨 · `split`(`pool` = 예시 후보 / `test` = 시험 글) · 글 · 벡터
- 뷰 `lab3.cls_proto` : 라벨마다 예시 후보 전체의 평균 = 프로토타입 `m` 과 그 길이 `m_norm`(응집도). 셀 9 의 분류 API 가 쓴다

In [ ]:
# 셀 3 · 테이블
if RESET:
    conn.execute("DROP TABLE IF EXISTS lab3.cls_items CASCADE")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.cls_items (
        id        bigserial PRIMARY KEY,
        ds        text NOT NULL,                 -- 데이터셋
        label     text NOT NULL,                 -- 클래스
        split     text NOT NULL CHECK (split IN ('pool', 'test')),
        text      text NOT NULL,
        embedding vector(4096))""")
conn.execute("CREATE INDEX IF NOT EXISTS cls_items_ds_label ON lab3.cls_items (ds, label)")
conn.execute("""
    CREATE OR REPLACE VIEW lab3.cls_proto AS
    SELECT ds, label, count(*) AS n, avg(embedding) AS m, vector_norm(avg(embedding)) AS m_norm
    FROM lab3.cls_items WHERE split = 'pool' AND embedding IS NOT NULL
    GROUP BY ds, label""")
sql("SELECT ds, count(*) AS 행 FROM lab3.cls_items GROUP BY ds ORDER BY ds")

### 셀 4 · 받기 · 표
데이터셋마다 원본을 받아(볼륨에 없을 때만) 글 · 라벨 표로 → 클래스마다 `POOL + TEST` 개를 뽑아 앞 `POOL` 개는 예시 후보, 나머지는 시험 글.
이미 테이블에 있는 데이터셋은 건너뛴다. 처음이면 특허 원본(약 1GB)을 받는 데 수 분 걸린다(빼려면 `DATASETS` 에서 patent 제거).

In [ ]:
# 셀 4 · 받기 · 표
def massive():
    """MASSIVE ko : 음성비서 명령 → 의도 60종. split = train / test"""
    parts = []
    for split in ["train", "test"]:
        d = pd.read_parquet(download(parquet_urls("mteb/amazon_massive_intent", "ko", split)[0], f"massive_{split}.parquet"))
        parts.append(d.assign(split=split)[["text", "label", "split"]])
    return clean(pd.concat(parts))


def ynat():
    """KLUE-YNAT : 뉴스 제목 → 주제 7종"""
    d = pd.read_parquet(download(parquet_urls("klue/klue", "ynat", "validation")[0], "ynat_validation.parquet"))
    names = ["IT과학", "경제", "사회", "생활문화", "세계", "스포츠", "정치"]
    return clean(d.assign(text=d["title"], label=d["label"].map(dict(enumerate(names))))[["text", "label"]])


def lbox():
    """LBox Open : 한국 판례 사실관계 → 죄명 (많은 순 12종)"""
    d = pd.read_parquet(download(parquet_urls("lbox/lbox_open", "casename_classification_plus", "train")[0], "lbox_casename_plus_train.parquet"))
    top = d["casename"].value_counts().head(12).index
    return clean(d[d["casename"].isin(top)].assign(text=lambda x: x["facts"], label=lambda x: x["casename"])[["text", "label"]])


OSHA_EVENT = {"1": "폭행 · 동물", "2": "교통 · 운반기계", "3": "화재 · 폭발", "4": "추락 · 넘어짐",
              "5": "유해물질 · 환경 노출", "6": "물체 · 장비 접촉(끼임 · 맞음)", "7": "과도한 힘 · 신체 반응"}


def osha():
    """OSHA 중대재해 보고 : 제조 · 건설 업종 사고 경위 → 사고 대분류 7종(사고 코드 앞자리)"""
    d = pd.read_parquet(download(parquet_urls("masonmarker/osha-severe-injury-reports-enriched", "default", "data")[0], "osha.parquet"))
    d = d[d["naics_sector_name"].isin(["Manufacturing", "Construction"]) & d["final_narrative"].notna()]
    d = d.assign(text=d["final_narrative"].str.strip(), label=d["event_code"].astype(str).str[0].map(OSHA_EVENT))
    return clean(d[d["label"].notna() & (d["text"].str.len() > 30)][["text", "label"]])


def nhtsa():
    """NHTSA 차량 결함 신고 → 결함 부품 대분류 5종"""
    d = pd.concat(pd.read_parquet(download(u, f"nhtsa_{i}.parquet")) for i, u in enumerate(all_parquet_urls("vic35get/nhtsa_complaints_dataset")))
    d = d.drop_duplicates("summary").assign(text=lambda x: x["summary"].str.strip())
    return clean(d[d["text"].str.len() > 30][["text", "label"]])


def product():
    """상품 설명 → 카테고리 33종"""
    d = pd.concat(pd.read_parquet(download(u, f"product_{i}.parquet")) for i, u in
                  enumerate(all_parquet_urls("Lezh1n/ecommerce-product-classification-by-categories")))
    return clean(d.drop_duplicates("text")[["text", "label"]])


def maib():
    """영국 MAIB 해양사고 보고 → 사고 유형"""
    d = pd.read_parquet(download(parquet_urls("baker-street/maib-incident-reports-5K", "default", "train")[0], "maib.parquet"))
    return clean(d[["text", "label"]])


def aviation():
    """항공 사고 원인 요약 → 원인 분류 (많은 순 8종)"""
    d = pd.read_parquet(download(parquet_urls("himaxym/aviation-safety-occurrences", "default", "train")[0], "aviation_0.parquet"))
    d = d[d["cause_summary"].fillna("").str.len() > 50]
    top = d["category"].value_counts().head(8).index
    return clean(d[d["category"].isin(top)].assign(text=lambda x: x["cause_summary"], label=lambda x: x["category"])[["text", "label"]])


PATENT_SECTION = {"a": "A 생활필수품", "b": "B 처리조작 · 운수", "c": "C 화학 · 야금", "d": "D 섬유 · 종이",
                  "e": "E 고정구조물(건설)", "f": "F 기계공학 · 조명 · 가열", "g": "G 물리학", "h": "H 전기"}


def patent():
    """BigPatent 시험 분할 : 특허 초록 → 기술 부문 A~H (부문마다 원본이 커서 처음 받을 때 수 분)"""
    parts = []
    for sec, name in PATENT_SECTION.items():
        a = pd.read_parquet(download(parquet_urls("NortheasternUniversity/big_patent", sec, "test")[0], f"patent_{sec}_test.parquet"), columns=["abstract"])
        parts.append(a.assign(text=a["abstract"], label=name)[["text", "label"]])
    return clean(pd.concat(parts))


def sample(ds):
    d = globals()[ds]()
    if ds == "massive":
        pool, test = per_label(d[d["split"] == "train"], POOL), per_label(d[d["split"] == "test"], TEST)
    else:
        d = d[d.groupby("label")["label"].transform("size") >= POOL + TEST]
        s = per_label(d, POOL + TEST)
        first = s.groupby("label").cumcount() < POOL
        pool, test = s[first], s[~first]
    return pd.concat([pool.assign(split="pool"), test.assign(split="test")])[["label", "split", "text"]].assign(ds=ds)


have = set(sql("SELECT DISTINCT ds FROM lab3.cls_items")["ds"]) if one("SELECT count(*) FROM lab3.cls_items") else set()
S = {}
for ds in DATASETS:
    if ds in have: print(f"{ds:9s} 테이블에 있음"); continue
    t = time.time(); S[ds] = sample(ds)
    print(f"{ds:9s} 클래스 {S[ds]['label'].nunique():3d} · 예시 후보 {(S[ds]['split'] == 'pool').sum():,} · 시험 {(S[ds]['split'] == 'test').sum():,} · {time.time() - t:.0f}s")

### 셀 5 · 적재

In [ ]:
# 셀 5 · 적재
for ds, d in S.items():
    conn.execute("""INSERT INTO lab3.cls_items (ds, label, split, text)
                    SELECT * FROM unnest(%s::text[], %s::text[], %s::text[], %s::text[])""",
                 [d[c].tolist() for c in ["ds", "label", "split", "text"]])
conn.execute("ANALYZE lab3.cls_items")
sql("""SELECT ds, count(DISTINCT label) AS 클래스, count(*) FILTER (WHERE split = 'pool') AS 예시_후보,
              count(*) FILTER (WHERE split = 'test') AS 시험, percentile_disc(0.5) WITHIN GROUP (ORDER BY length(text)) AS 글_길이_중앙
       FROM lab3.cls_items GROUP BY ds ORDER BY ds""")

### 셀 6 · 임베딩
짧은 글(명령 · 제목)은 빠르고, 판례 · 특허처럼 긴 글은 느리다. 실습 4 를 먼저 했다면 겹치는 글(판례 · 사고 · 결함 신고)은 캐시에서 채워진다.

In [ ]:
# 셀 6 · 임베딩
embed_table("lab3.cls_items")

### 셀 7 · 불러오기 · 채점 도구
- `run(ds, k, r)` : 클래스마다 예시 후보에서 k 개를 뽑아 평균(길이 그대로 = 내적용, 길이 1 = 코사인용) → 시험 글을 가장 가까운 프로토타입으로

In [ ]:
# 셀 7 · 불러오기 · 채점 도구
data = {}
for ds in DATASETS:
    if not one("SELECT count(*) FROM lab3.cls_items WHERE ds = %s", (ds,)): print(ds, ": 테이블에 없음 → 셀 4 부터"); continue
    data[ds] = load("SELECT label, split, text, embedding FROM lab3.cls_items WHERE ds = %s ORDER BY id", (ds,))
print({k: f"{len(v[0]):,}행 · 클래스 {v[0]['label'].nunique()}" for k, v in data.items()})


def run(ds, k, r):
    d, X = data[ds]
    classes = sorted(d["label"].unique())
    pool, test = np.flatnonzero(d["split"].values == "pool"), np.flatnonzero(d["split"].values == "test")
    lab = d["label"].values
    M = np.stack([X[r.choice(pc, min(k, len(pc)), replace=False)].mean(axis=0) for pc in (pool[lab[pool] == c] for c in classes)])
    y = lab[test]
    cos = np.array(classes)[(X[test] @ unit(M).T).argmax(1)]
    dot = np.array(classes)[(X[test] @ M.T).argmax(1)]
    return cos == y, dot == y, len(classes)                        # 시험 글마다 맞혔나 (코사인 · 내적)

### 셀 8 · 한 예시 — 고객 문의(한국어)
의도마다 예시 3개 평균을 프로토타입으로. 시험 문장이 어느 의도와 가장 가까운지(코사인 상위 3).

In [ ]:
# 셀 8 · 고객 문의 한 예시
d, X = data["massive"]
r = np.random.default_rng(SEED)
classes = sorted(d["label"].unique())
pool, test = np.flatnonzero(d["split"].values == "pool"), np.flatnonzero(d["split"].values == "test")
ex = {c: r.choice(pool[d["label"].values[pool] == c], 3, replace=False) for c in classes}
M = unit(np.stack([X[ex[c]].mean(axis=0) for c in classes]))
print("예시 (의도 3개만) :")
for c in classes[:3]: print(f"  {c:18s}", " / ".join(d["text"].values[ex[c]]))
rows = []
for i in r.choice(test, 8, replace=False):
    s = X[i] @ M.T; top = np.argsort(-s)[:3]
    rows.append({"문장": d["text"][i], "정답": d["label"][i], "예측": classes[top[0]],
                 "상위 3 (코사인)": " · ".join(f"{classes[t]} {s[t]:.2f}" for t in top)})
pd.DataFrame(rows)

### 셀 9 · 분류 API — 새 문장
서비스에서는 프로토타입을 표(뷰)로 두고 새 글이 들어올 때마다 SQL 한 번으로 분류한다.
`lab3.cls_proto`(라벨마다 예시 후보 `POOL` 개의 평균)를 프로토타입으로, 처음 보는 문장을 임베딩해 가장 가까운 라벨 3개를 본다.
- 질의 문장도 지시문 없이 임베딩한다(프로토타입이 지시문 없는 글의 평균이므로)

In [ ]:
# 셀 9 · 분류 API
NEW = [("massive", "내일 아침 일곱 시에 깨워 줘"),
       ("massive", "거실 불 좀 꺼 줄래"),
       ("ynat", "반도체 수출 석 달 연속 증가…대중 수출은 감소"),
       ("lbox", "피고인은 혈중알코올농도 0.12%의 술에 취한 상태로 승용차를 운전하였다."),
       ("osha", "An employee's hand was caught in the rollers of a conveyor while clearing a jam."),
       ("nhtsa", "When I pressed the brake pedal at a stop light the pedal went to the floor and the car did not stop."),
       ("aviation", "The pilot did not maintain adequate airspeed during the final approach, resulting in a stall.")]
CLASSIFY = """SELECT label AS 라벨, round((1 - (m <=> %(v)s))::numeric, 3) AS 코사인
              FROM lab3.cls_proto WHERE ds = %(ds)s ORDER BY m <=> %(v)s LIMIT 3"""
NEW = [(ds, s) for ds, s in NEW if ds in data]
rows = []
for (ds, s), v in zip(NEW, embed([s for _, s in NEW])):
    top = sql(CLASSIFY, {"v": v, "ds": ds})
    rows.append({"데이터": ds, "새 문장": s, "1위": top["라벨"][0],
                 "상위 3 (코사인)": " · ".join(f"{a} {b}" for a, b in zip(top["라벨"], top["코사인"]))})
pd.DataFrame(rows)

### 셀 10 · 예시 수별 정확도 — 9개 산업
클래스당 예시 k 개 · `REPEAT` 번 다시 뽑아 평균. 점선 = 찍기(1 ÷ 클래스 수). 시험 글이 클래스당 `TEST` 개라 1~2%p 는 흔들린다.

In [ ]:
# 셀 10 · 예시 수별 정확도
t = time.time(); rows, ACC = [], {}
for ds in data:
    for k in SHOTS:
        acc = [run(ds, k, np.random.default_rng(SEED + i)) for i in range(REPEAT)]
        ACC[ds, k] = (np.mean([a[0] for a in acc], axis=0), np.mean([a[1] for a in acc], axis=0))   # 글마다 REPEAT 번 중 맞힌 비율
        rows.append({"데이터": ds, "산업": INDUSTRY[ds], "예시 수": k, "정확도": ACC[ds, k][0].mean(),
                     "내적 정확도": ACC[ds, k][1].mean(), "찍기": 1 / acc[0][2]})
R10 = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(11, 4.2))
for ds, g in R10.groupby("데이터", sort=False):
    ax.plot(g["예시 수"], g["정확도"], "-o", label=f"{ds} ({INDUSTRY[ds]})")
ax.set_xscale("log"); ax.set_xticks(SHOTS, SHOTS); ax.set_xlabel("클래스당 예시 수"); ax.set_ylabel("정확도"); ax.set_ylim(0, 1)
ax.legend(fontsize=8, ncol=3, loc="lower right"); ax.set_title("프로토타입 분류 정확도 (코사인)"); plt.show()
print(f"{time.time() - t:.0f}s")
R10.pivot(index="데이터", columns="예시 수", values="정확도").join(R10.groupby("데이터")["찍기"].first()).round(3)

### 셀 11 · 코사인 vs 내적
장표 21 : 클래스마다 응집도(‖M‖)가 다르면, 내적으로 비교할 때 잘 모인 클래스(긴 M)가 유리 → 코사인으로.
- 예시 3개일 때 두 방식의 정확도 · 차이의 95% 신뢰구간(시험 글 단위) · 클래스 ‖M‖ 의 최소 ~ 최대 (‖M‖ = `lab3.cls_proto.m_norm`, 예시 후보 전체 평균의 길이)
- ‖M‖ 의 폭이 좁은 데이터는 두 방식의 차이도 작다

In [ ]:
# 셀 11 · 코사인 vs 내적
rows = []
for ds in data:
    norms = sql("SELECT m_norm FROM lab3.cls_proto WHERE ds = %s", (ds,))["m_norm"]
    g = R10[(R10["데이터"] == ds) & (R10["예시 수"] == 3)].iloc[0]
    rows.append({"데이터": ds, "코사인": g["정확도"], "내적": g["내적 정확도"], "차이 [95% 구간]": ci(ACC[ds, 3][0] - ACC[ds, 3][1]),
                 "클래스 ‖M‖ 최소": norms.min(), "최대": norms.max()})
pd.DataFrame(rows).set_index("데이터").round(3)

### 셀 12 · "기타" 거르기 — 처음 보는 클래스
클래스의 30% 를 "처음 보는 문의"로 숨기고, 나머지로만 프로토타입(예시 5개)을 만든다.
가장 가까운 프로토타입과의 코사인이 기준값보다 낮으면 "기타".
- 기준값 = **아는 클래스 글의 95% 를 통과시키는 값**(아는 문의는 5% 만 잘못 거름)
- 그때 처음 보는 글을 몇 % 걸러내나
- 점수 네 가지 비교 : 기준(가장 가까운 프로토타입과의 코사인) · 공통 성분 제거 뒤 코사인(아는 클래스 예시 후보의 평균을 빼고 길이 1) · 1위 − 2위 점수 차 · 가장 가까운 예시 글(아는 클래스 예시 후보 전부)과의 코사인

In [ ]:
# 셀 12 · 기타 거르기
rows = []
for ds in data:
    d, X = data[ds]
    r = np.random.default_rng(SEED)
    lab = d["label"].values
    classes = np.array(sorted(set(lab)))
    unk = r.choice(classes, max(1, int(len(classes) * 0.3)), replace=False)
    known = np.setdiff1d(classes, unk)
    pool, test = np.flatnonzero(d["split"].values == "pool"), np.flatnonzero(d["split"].values == "test")
    picks = [r.choice(pool[lab[pool] == c], min(5, (lab[pool] == c).sum()), replace=False) for c in known]
    M = unit(np.stack([X[p].mean(axis=0) for p in picks]))
    kp = pool[np.isin(lab[pool], known)]                               # 아는 클래스의 예시 후보 전부
    Xc = unit(X - X[kp].mean(axis=0)); Mc = unit(np.stack([Xc[p].mean(axis=0) for p in picks]))
    S = X[test] @ M.T; srt = np.sort(S, axis=1)
    is_unk = np.isin(lab[test], unk)
    score = {"기준 : 최고 코사인": srt[:, -1], "공통 성분 제거 뒤 코사인": (Xc[test] @ Mc.T).max(1),
             "1위 − 2위 차": srt[:, -1] - srt[:, -2], "가장 가까운 예시 글": (X[test] @ X[kp].T).max(1)}
    s = score["기준 : 최고 코사인"]; th = np.quantile(s[~is_unk], 0.05)
    row = {"데이터": ds, "기준 코사인": th, "아는 글 코사인 중앙": np.median(s[~is_unk]), "처음 보는 글 코사인 중앙": np.median(s[is_unk])}
    for name, v in score.items():                                       # 점수마다 아는 글 95% 통과 기준 → 처음 보는 글 걸러낸 비율
        row[name] = (v[is_unk] < np.quantile(v[~is_unk], 0.05)).mean()
    rows.append(row)
R12 = pd.DataFrame(rows).set_index("데이터").round(3)
R12.loc["평균"] = R12.mean().round(3)
R12

### 셀 13 · 정리
- 셀 10 : 예시 몇 개로 어느 산업에서 쓸 만한가 · 셀 11 : 코사인이 내적보다 나은가
- 셀 12 : 가장 가까운 프로토타입 코사인만으로는 처음 보는 유형을 잘 못 거른다. 공통 성분을 빼면 나아지는 데이터가 많다(전체 규모 평균 0.20 → 0.29). 1위 − 2위 차는 오히려 나쁘다. 기준값은 데이터마다 다르다 → 내 데이터로 잰다

In [ ]:
# 셀 13 · 정리
print(f"클래스당 예시 후보 {POOL} · 시험 {TEST}")
R10.pivot(index="데이터", columns="예시 수", values="정확도").round(3).join(R12[["기준 : 최고 코사인", "공통 성분 제거 뒤 코사인"]].rename(columns=lambda c: "걸러냄 · " + c))